In [1]:
# re: used for text cleaning with regular expressions
# (remove HTML entities, URLs, special characters)
import re

# emoji: used to detect and replace emojis in text
# helps reduce noise while keeping emotional signals
import emoji

# numpy: used for numerical operations and feature arrays
import numpy as np

# pandas: used to load and manage datasets in DataFrame format
import pandas as pd

# word_tokenize: used to tokenize Thai text into words
# Thai language does not have spaces between words
from pythainlp.tokenize import word_tokenize

# train_test_split: used to split data into training and testing sets
from sklearn.model_selection import train_test_split

# TfidfVectorizer: converts text into numerical features using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# evaluation metrics for model performance
# accuracy_score: overall accuracy
# classification_report: precision, recall, F1-score
# confusion_matrix: shows prediction errors for each class
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# LabelEncoder: converts text labels (positive, neutral, negative)
# into numeric labels for machine learning models
from sklearn.preprocessing import LabelEncoder

# hstack: used to combine multiple feature matrices horizontally
from scipy.sparse import hstack

# LightGBM: gradient boosting model for high-performance text classification
import lightgbm as lgb

In [2]:
def clean_text(text):
    # convert to string
    text = str(text)

    # remove HTML entities (e.g. &#128591;)
    text = re.sub(r'&#\d+;', ' ', text)

    # replace URLs with a special token
    text = re.sub(r'http\S+|www\S+', ' <URL> ', text)

    # replace user mentions
    text = re.sub(r'@\S+', ' <USER> ', text)

    # replace emojis with a token
    text = emoji.replace_emoji(text, replace=' <EMOJI> ')

    # keep only Thai, English characters and numbers
    text = re.sub(r'[^ก-๙a-zA-Z0-9\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [3]:
def thai_tokenizer(text):
    # tokenize Thai text into words
    return word_tokenize(text, engine="newmm")

In [7]:
df = pd.read_json("../../dataset/train_sentiment.json")

# if 'text' column is missing, fix orientation
if "text" not in df.columns:
    df = df.transpose().reset_index(drop=True)

print(df.columns)
df.head()

Index(['text', 'sentiment'], dtype='object')


,text,sentiment
0,มาตามนัด 🚮🗑️ . . #เขตพญาไท #กรุงเทพมหานคร @สาย...,positive
1,&#9888; แยกราชเทวีจะรื้อน้ำพุกี่โมง? &#128591;...,negative
2,"&#128308;สด!""ไอซ์ รักชนก"" แถลงข่าวการประมูลคลื...",negative
3,&#127808;#เขตหนองจอก >>ติดตามการดำเนินงานโครงก...,neutral
4,ไทวัสดุ ส่งช่างมือ 1 วีฟิกซ์ ร่วมฟื้นฟู ปรับภู...,positive


In [8]:
# clean text
df["clean_text"] = df["text"].apply(clean_text)

# input and target
X = df["clean_text"]
y = df["sentiment"]

# encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# show label mapping
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


In [9]:
# split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [10]:
# word-level TF-IDF features
tfidf_word = TfidfVectorizer(
    tokenizer=thai_tokenizer,
    ngram_range=(1, 2),
    min_df=3,
    max_features=50000
)

X_word_train = tfidf_word.fit_transform(X_train)
X_word_test = tfidf_word.transform(X_test)

C:\Users\watta\anaconda3\envs\thai-sentiment\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
# character-level TF-IDF features
tfidf_char = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3,
    max_features=30000
)

X_char_train = tfidf_char.fit_transform(X_train)
X_char_test = tfidf_char.transform(X_test)

In [12]:
# combine word and character features
X_train_final = hstack([X_word_train, X_char_train])
X_test_final = hstack([X_word_test, X_char_test])

In [13]:
# LightGBM model
model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(le.classes_),
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# train the model
model.fit(X_train_final, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.886854 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2735866
[LightGBM] [Info] Number of data points in the train set: 7200, number of used features: 43923
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.05
,n_estimators,600
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [14]:
# predict on test set
y_pred = model.predict(X_test_final)

# accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# precision, recall, F1-score
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

C:\Users\watta\anaconda3\envs\thai-sentiment\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.7838888888888889

Classification Report:
              precision    recall  f1-score   support

    negative       0.83      0.84      0.83       600
     neutral       0.72      0.69      0.70       600
    positive       0.80      0.82      0.81       600

    accuracy                           0.78      1800
   macro avg       0.78      0.78      0.78      1800
weighted avg       0.78      0.78      0.78      1800

Confusion Matrix:
[[504  82  14]
 [ 78 412 110]
 [ 26  79 495]]


In [15]:
import joblib

# save LightGBM model
joblib.dump(model, "lgb_model.pkl")

# save TF-IDF vectorizers
joblib.dump(tfidf_word, "tfidf_word.pkl")
joblib.dump(tfidf_char, "tfidf_char.pkl")

# save label encoder
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']